In [0]:
import logging
from datetime import datetime
from pyspark.sql import SparkSession

# Configuração do Logger Estruturado (Ponto 8)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] [%(name)s] - %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("GoldFatoTransacaoPipeline")

spark = SparkSession.builder.getOrCreate()

def registrar_execucao_controle(spark: SparkSession, batch_id: str, tabela: str, lidas: int, rejeitadas: int, status: str, inicio: datetime):
    """Grava as métricas de execução em uma tabela Delta de auditoria."""
    fim = datetime.now()
    duracao_segundos = (fim - inicio).total_seconds()
    
    dados_log = [(batch_id, tabela, lidas, rejeitadas, status, inicio, fim, duracao_segundos)]
    colunas = ["batch_id", "tabela", "linhas_lidas", "linhas_rejeitadas", "status", "timestamp_inicio", "timestamp_fim", "duracao_segundos"]
    
    df_log = spark.createDataFrame(dados_log, colunas)
    df_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("workspace.default.controle_execucoes")
    logger.info(f"Métricas de auditoria gravadas para a tabela: {tabela} [Status: {status}]")

def build_gold_fato_transacao(batch_id: str = "batch_manual_001"):
    inicio_execucao = datetime.now()
    logger.info("Iniciando a construção da gold_fato_transacao...")
    
    query = """
    CREATE OR REPLACE TABLE workspace.default.gold_fato_transacao
    CLUSTER BY (data_transacao, id_cartao) AS
    
    -- 1. CTE de Transações base
    WITH transacoes AS (
        SELECT * FROM workspace.default.silver_transacoes
    ),
    
    -- 2. CTE de Estornos (Preparação para anular o valor líquido)
    estornos AS (
        SELECT id_transacao, data_estorno, motivo 
        FROM workspace.default.silver_estornos
    ),
    
    -- 3. CTE do Histórico de Cartões (Tratando valid_to nulo como o "infinito")
    cartoes_hist AS (
        SELECT 
            id_cartao, 
            id_conta, 
            tipo_cartao, 
            status_cartao,
            valid_from, 
            COALESCE(valid_to, '9999-12-31 23:59:59') AS valid_to
        FROM workspace.default.silver_cartoes
    ),
    
    -- 4. CTE Final integrando as regras de negócio
    fato_enriquecida AS (
        SELECT 
            t.id_transacao,
            t.id_cartao,
            c.id_conta,
            t.data_transacao,
            t.mcc,
            t.estabelecimento,
            t.canal,
            
            -- Regra de Negócio: Se tem estorno, o valor líquido é zerado
            t.valor AS valor_bruto,
            CASE WHEN e.id_transacao IS NOT NULL THEN 0 ELSE t.valor END AS valor_liquido,
            
            -- Flags analíticas
            CASE WHEN e.id_transacao IS NOT NULL THEN TRUE ELSE FALSE END AS is_estornada,
            e.motivo AS motivo_estorno,
            
            -- Snapshot do status do cartão no momento da compra
            c.status_cartao AS status_cartao_no_momento
            
        FROM transacoes t
        
        -- Join com Estornos
        LEFT JOIN estornos e 
            ON t.id_transacao = e.id_transacao
            
        -- POINT-IN-TIME JOIN (SCD2): Pega a versão exata do cartão na data da transação
        LEFT JOIN cartoes_hist c 
            ON t.id_cartao = c.id_cartao 
            AND t.data_transacao >= c.valid_from 
            AND t.data_transacao < c.valid_to
    )
    
    SELECT * FROM fato_enriquecida;
    """
    
    try:
        spark.sql(query)
        logger.info("Tabela workspace.default.gold_fato_transacao criada com sucesso.")
        
        # Aplicando OPTIMIZE para compactação e manutenção do Liquid Clustering (Ponto 9)
        logger.info("Executando OPTIMIZE na tabela gold_fato_transacao...")
        spark.sql("OPTIMIZE workspace.default.gold_fato_transacao")
        logger.info("OPTIMIZE executado com sucesso.")
        
        # Captura volumetria gerada para auditoria
        total_linhas = spark.read.table("workspace.default.gold_fato_transacao").count()
        
        # Registra sucesso na tabela de controle
        registrar_execucao_controle(
            spark=spark, 
            batch_id=batch_id, 
            tabela="gold_fato_transacao", 
            lidas=total_linhas, 
            rejeitadas=0, 
            status="SUCESSO", 
            inicio=inicio_execucao
        )
        
    except Exception as e:
        logger.error(f"Erro ao processar gold_fato_transacao: {str(e)}")
        registrar_execucao_controle(
            spark=spark, 
            batch_id=batch_id, 
            tabela="gold_fato_transacao", 
            lidas=0, 
            rejeitadas=0, 
            status="ERRO", 
            inicio=inicio_execucao
        )
        raise e

# Executa a criação da fato otimizada
build_gold_fato_transacao(batch_id="batch_2026_07_26")

# Exibe uma amostra para validarmos
display(spark.sql("""
    SELECT id_transacao, data_transacao, valor_bruto, valor_liquido, is_estornada, status_cartao_no_momento
    FROM workspace.default.gold_fato_transacao 
    ORDER BY is_estornada DESC 
    LIMIT 5
"""))